# 🏥 AiRadiology - Chest X-ray Pneumonia Detection
## Google Colab Training Notebook (GPU T4)

**Model:** ResNet50 + Custom Head (Transfer Learning)
**Dataset:** Chest X-ray (Pneumonia vs Normal)
**Expected Accuracy:** 95%+
**Training Time:** ~30 minutes on T4 GPU

---
### ⚡ Steps:
1. Upload dataset to Google Drive
2. Run all cells
3. Download trained model
4. Use in your web app

## 📌 Step 1: Check GPU

In [ ]:
# Check GPU availability
import tensorflow as tf
print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

# Show GPU info
!nvidia-smi

## 📌 Step 2: Mount Google Drive & Upload Dataset

**Before running this cell:**
1. Go to Google Drive
2. Create folder: `MyDrive/chest_xray`
3. Upload your `chest_xray` folder there
4. Structure should be:
   ```
   MyDrive/chest_xray/
   ├── train/
   │   ├── NORMAL/
   │   └── PNEUMONIA/
   └── test/
       ├── NORMAL/
       └── PNEUMONIA/
   ```

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check dataset structure
import os

DATASET_PATH = '/content/drive/MyDrive/chest_xray'

# Count images
for split in ['train', 'test']:
    for cls in ['NORMAL', 'PNEUMONIA']:
        path = os.path.join(DATASET_PATH, split, cls)
        if os.path.exists(path):
            count = len(os.listdir(path))
            print(f'{split}/{cls}: {count} images')
        else:
            print(f'❌ Path not found: {path}')

## 📌 Step 3: Install & Import Libraries

In [ ]:
# Install dependencies
!pip install -q tensorflow numpy matplotlib scikit-learn

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print('✅ All libraries imported!')

## 📌 Step 4: Configuration

In [ ]:
# ===== CONFIGURATION =====
IMG_SIZE    = 224
BATCH_SIZE  = 32
EPOCHS      = 20
LR          = 0.0001

TRAIN_DIR   = '/content/drive/MyDrive/chest_xray/train'
TEST_DIR    = '/content/drive/MyDrive/chest_xray/test'
MODEL_PATH  = '/content/drive/MyDrive/pneumonia_model.h5'
TFLITE_PATH = '/content/drive/MyDrive/pneumonia_model.tflite'

print('=' * 50)
print('🏥 AiRadiology Training Configuration')
print('=' * 50)
print(f'Image Size:  {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch Size:  {BATCH_SIZE}')
print(f'Epochs:      {EPOCHS}')
print(f'Learning Rate: {LR}')
print('=' * 50)

## 📌 Step 5: Data Preparation

In [ ]:
# Data augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    validation_split=0.2
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Load training data
print('📂 Loading training data...')
train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training',
    shuffle=True
)

print('📂 Loading validation data...')
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation',
    shuffle=True
)

print('📂 Loading test data...')
test_gen = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

print(f'\n✅ Classes: {train_gen.class_indices}')
print(f'Training: {train_gen.samples} | Validation: {val_gen.samples} | Test: {test_gen.samples}')

## 📌 Step 6: Visualize Sample Data

In [ ]:
# Show sample X-ray images
images, labels = next(train_gen)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i, ax in enumerate(axes.flat):
    if i < len(images):
        ax.imshow(images[i])
        label = 'PNEUMONIA' if labels[i] == 1 else 'NORMAL'
        color = 'red' if labels[i] == 1 else 'green'
        ax.set_title(label, color=color, fontweight='bold')
        ax.axis('off')

plt.suptitle('Sample X-ray Images', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 📌 Step 7: Build Model (ResNet50 Transfer Learning)

In [ ]:
print('🏗️ Building ResNet50 model...')

# Base model - pretrained on ImageNet
base = ResNet50(
    include_top=False,
    weights='imagenet',
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)
base.trainable = False  # Freeze base

# Build full model
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(512, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = keras.Model(inputs, outputs)

# Compile
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall'),
             keras.metrics.AUC(name='auc')]
)

print(f'✅ Model built! Parameters: {model.count_params():,}')
model.summary()

## 📌 Step 8: Train Model 🚀

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        MODEL_PATH,
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('🚀 Training started...')
print('=' * 50)

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print('\n✅ Training Complete!')

## 📌 Step 9: Fine-tuning (Optional but improves accuracy)

In [ ]:
# Unfreeze last 30 layers for fine-tuning
print('🔧 Fine-tuning top layers...')

base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR / 10),
    loss='binary_crossentropy',
    metrics=['accuracy',
             keras.metrics.Precision(name='precision'),
             keras.metrics.Recall(name='recall'),
             keras.metrics.AUC(name='auc')]
)

history_ft = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

print('✅ Fine-tuning Complete!')

## 📌 Step 10: Evaluate Model

In [ ]:
print('📊 Evaluating on test set...')
results = model.evaluate(test_gen, verbose=1)

print('\n' + '=' * 50)
print('🎯 FINAL TEST RESULTS:')
print('=' * 50)
print(f'Accuracy:  {results[1]*100:.2f}%')
print(f'Precision: {results[2]*100:.2f}%')
print(f'Recall:    {results[3]*100:.2f}%')
print(f'AUC:       {results[4]*100:.2f}%')
print('=' * 50)

# Predictions
test_gen.reset()
preds = (model.predict(test_gen) > 0.5).astype(int).flatten()
true_labels = test_gen.classes

# Classification report
print('\n📋 Classification Report:')
print(classification_report(true_labels, preds, target_names=['NORMAL', 'PNEUMONIA']))

## 📌 Step 11: Plot Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(history.history['accuracy'],     label='Train',      linewidth=2, color='#00E5FF')
axes[0].plot(history.history['val_accuracy'], label='Validation', linewidth=2, color='#6C63FF')
axes[0].set_title('Model Accuracy', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Loss
axes[1].plot(history.history['loss'],     label='Train',      linewidth=2, color='#00E5FF')
axes[1].plot(history.history['val_loss'], label='Validation', linewidth=2, color='#6C63FF')
axes[1].set_title('Model Loss', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('AiRadiology Training History', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/training_history.png', dpi=200)
plt.show()
print('✅ Plot saved!')

## 📌 Step 12: Confusion Matrix

In [ ]:
cm = confusion_matrix(true_labels, preds)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['NORMAL', 'PNEUMONIA'],
            yticklabels=['NORMAL', 'PNEUMONIA'])
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/confusion_matrix.png', dpi=200)
plt.show()

## 📌 Step 13: Export Models

In [ ]:
# Save Keras model (.h5)
model.save(MODEL_PATH)
print(f'✅ Keras model saved: {MODEL_PATH}')

# Convert to TensorFlow Lite (for mobile/web)
print('\n📦 Converting to TFLite...')
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

print(f'✅ TFLite model saved: {TFLITE_PATH}')

# File sizes
h5_size     = os.path.getsize(MODEL_PATH) / (1024*1024)
tflite_size = os.path.getsize(TFLITE_PATH) / (1024*1024)

print(f'\n📁 Keras model size:  {h5_size:.1f} MB')
print(f'📁 TFLite model size: {tflite_size:.1f} MB')

print('\n' + '=' * 50)
print('🎉 TRAINING COMPLETE!')
print('=' * 50)
print('Files saved to Google Drive:')
print(f'  📦 pneumonia_model.h5')
print(f'  📦 pneumonia_model.tflite')
print(f'  📊 training_history.png')
print(f'  📊 confusion_matrix.png')
print('\nDownload these files and use in your web app!')

## 📌 Step 14: Test with Sample Images

In [ ]:
from PIL import Image
import numpy as np

def predict_xray(image_path):
    """Predict pneumonia from X-ray image"""
    img = Image.open(image_path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    arr = np.array(img) / 255.0
    arr = np.expand_dims(arr, 0)
    pred = model.predict(arr, verbose=0)[0][0]
    return ('PNEUMONIA', pred * 100) if pred > 0.5 else ('NORMAL', (1-pred) * 100)

# Test with real images
import glob
normal_samples    = glob.glob(TEST_DIR + '/NORMAL/*.jpeg')[:3]
pneumonia_samples = glob.glob(TEST_DIR + '/PNEUMONIA/*.jpeg')[:3]
all_samples = normal_samples + pneumonia_samples

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
for i, (path, ax) in enumerate(zip(all_samples, axes.flat)):
    img     = Image.open(path).convert('RGB').resize((IMG_SIZE, IMG_SIZE))
    result, conf = predict_xray(path)
    true_label   = 'NORMAL' if 'NORMAL' in path else 'PNEUMONIA'
    correct      = result == true_label

    ax.imshow(img)
    ax.axis('off')
    color = 'green' if correct else 'red'
    ax.set_title(f'True: {true_label}\nPred: {result} ({conf:.1f}%)\n{"✅" if correct else "❌"}',
                 color=color, fontweight='bold', fontsize=10)

plt.suptitle('AiRadiology - Test Predictions', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()
print('🎯 Predictions complete!')